# Moving-Average Cross Meta-Labeling

Build a trend-following primary model from moving-average crosses, derive AFML meta-labels with `pt_sl=[1, 2]` and a one-day vertical barrier, then train a random forest to decide whether to trade. The primary model decides the side; the random forest only decides bet/no bet.

In [ ]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from afml.bars import dollar_bars
from afml.data import read_tick_csv
from afml.labeling import add_vertical_barrier, drop_labels, get_bins, get_events
from afml.sampling import daily_volatility, target_from_events
from afml.strategies import moving_average_cross_side

In [ ]:
RAW_PATH = PROJECT_ROOT / "data" / "Binance" / "spot" / "daily" / "trades" / "BTCUSDT" / "BTCUSDT-trades-2026-06-03.csv"
NORMALIZED_PATH = PROJECT_ROOT / "data" / "processed" / "binance" / "trades" / "BTCUSDT" / "BTCUSDT-trades-2026-06-03-normalized.csv"

if not NORMALIZED_PATH.exists():
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "prepare_binance_trades.py"), str(RAW_PATH)],
        cwd=PROJECT_ROOT,
        check=True,
    )

NORMALIZED_PATH

In [ ]:
NROWS = None
DOLLAR_BAR_COUNT = 500

raw_ticks = read_tick_csv(NORMALIZED_PATH, nrows=NROWS)
dollar_threshold = raw_ticks["dollar_value"].sum() / DOLLAR_BAR_COUNT
bars = dollar_bars(raw_ticks, threshold=dollar_threshold)
close = bars["close"]

pd.Series(
    {
        "ticks": len(raw_ticks),
        "bars": len(bars),
        "start": close.index.min(),
        "end": close.index.max(),
        "dollar_threshold": dollar_threshold,
    }
)

In [ ]:
FAST_WINDOW = 20
SLOW_WINDOW = 50

side = moving_average_cross_side(close, fast_window=FAST_WINDOW, slow_window=SLOW_WINDOW)
t_events = side.index

side.value_counts().rename("count")

In [ ]:
daily_vol = daily_volatility(close, span=100)

if daily_vol.empty:
    raise ValueError(
        "daily_volatility is empty. Snippet 3.1 needs observations at least one calendar day apart; "
        "load multiple days of bars before deriving these meta-labels."
    )

trgt = target_from_events(daily_vol, t_events, min_ret=0)
vertical_barriers = add_vertical_barrier(trgt.index, close, num_days=1)

meta_events = get_events(
    close=close,
    t_events=trgt.index,
    pt_sl=[1, 2],
    trgt=trgt,
    min_ret=0,
    t1=vertical_barriers,
    side=side,
)
labels = get_bins(meta_events, close)
labels = drop_labels(labels, min_pct=0.05)

labels["bin"].value_counts(normalize=True).rename("class_fraction")

In [ ]:
features = pd.DataFrame(index=close.index)
fast_ma = close.rolling(FAST_WINDOW).mean()
slow_ma = close.rolling(SLOW_WINDOW).mean()
features["ma_spread"] = fast_ma / slow_ma - 1
features["ret_1"] = close.pct_change()
features["ret_5"] = close.pct_change(5)
features["vol_20"] = close.pct_change().rolling(20).std()
features["side"] = side
features["volume"] = bars["volume"]
features["dollar_value"] = bars["dollar_value"]
features = features.shift(1)

sample = features.join(labels["bin"], how="inner").dropna()
X = sample.drop(columns="bin")
y = sample["bin"].astype(int)

pd.Series({"samples": len(sample), "features": X.shape[1], "positive_fraction": y.mean()})

In [ ]:
if y.nunique() < 2:
    raise ValueError("Need at least two meta-label classes after filtering to train the classifier.")

split = int(len(sample) * 0.7)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

rf = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=5,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

proba = rf.predict_proba(X_test)[:, list(rf.classes_).index(1)]
pred = (proba >= 0.5).astype(int)

print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, digits=3))
if y_test.nunique() == 2:
    print("roc_auc", roc_auc_score(y_test, proba))

In [ ]:
feature_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
feature_importance